In [1]:
# ============================================================
# ANN REGRESSION - INSURANCE EXPENSE PREDICTION
# 63 EXPERIMENTS | EXACTLY 13 PLOT IMAGES
# ============================================================

import os
import time
import warnings
import psutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import (
    r2_score, mean_squared_error, mean_absolute_error
)

warnings.filterwarnings("ignore", category=UserWarning)

# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

RANDOM_STATE = 42
TEST_SIZE = 0.20
VALIDATION_SIZE_FROM_TRAIN = 0.25

HIDDEN_LAYER_COUNTS = list(range(1, 8))  # 1 to 7 hidden layers
NEURONS_PER_LAYER = 64

ACTIVATION_FUNCTIONS = ["relu", "tanh", "logistic"]
LEARNING_RATES = [0.001, 0.01, 0.1]

MAX_ITER = 500
OUTPUT_DIR = "ANN_Regression_13_Plots"
os.makedirs(OUTPUT_DIR, exist_ok=True)

plt.rcParams["font.size"] = 10
plt.rcParams["axes.titlesize"] = 12
plt.rcParams["axes.labelsize"] = 10

# ------------------------------------------------------------
# 2. LOAD AND INSPECT DATASET
# ------------------------------------------------------------

DATA_PATH = "insurance.csv"

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        "insurance.csv was not found. Place it in the same folder "
        "as this Python script."
    )

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
print("\nMissing values:\n", df.isnull().sum())

if "expenses" not in df.columns:
    raise ValueError("The target column 'expenses' was not found.")

if df.isnull().any().any():
    raise ValueError("Missing values detected. Handle them before training.")

X = df.drop(columns=["expenses"])
y = df["expenses"]

numeric_features = ["age", "bmi", "children"]
categorical_features = ["sex", "smoker", "region"]

# ------------------------------------------------------------
# 3. TRAIN / VALIDATION / TEST SPLIT
# ------------------------------------------------------------
# 60% training, 20% validation, 20% final test.
# All 63 configurations are compared using validation data.
# The test set is used only once after selecting the best model.

X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val,
    y_train_val,
    test_size=VALIDATION_SIZE_FROM_TRAIN,
    random_state=RANDOM_STATE
)

print("\nTraining samples:", len(X_train))
print("Validation samples:", len(X_val))
print("Testing samples:", len(X_test))


Dataset shape: (1338, 7)

Missing values:
 age         0
sex         0
bmi         0
children    0
smoker      0
region      0
expenses    0
dtype: int64

Training samples: 802
Validation samples: 268
Testing samples: 268


In [2]:
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", StandardScaler(), numeric_features),
        ("categorical",
         OneHotEncoder(drop="first", handle_unknown="ignore"),
         categorical_features)
    ]
)

X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed = preprocessor.transform(X_val)
X_train_val_processed = preprocessor.fit_transform(X_train_val)
X_test_processed = preprocessor.transform(X_test)

# Standardize target values for stable ANN training.
y_scaler = StandardScaler()
y_train_scaled = y_scaler.fit_transform(
    np.asarray(y_train).reshape(-1, 1)
).ravel()

y_train_val_scaled = y_scaler.fit_transform(
    np.asarray(y_train_val).reshape(-1, 1)
).ravel()

In [3]:
process = psutil.Process(os.getpid())
experiment_results = []
loss_curves = {}
validation_predictions = {}

total_experiments = (
    len(HIDDEN_LAYER_COUNTS)
    * len(ACTIVATION_FUNCTIONS)
    * len(LEARNING_RATES)
)

run_number = 0
total_start = time.perf_counter()

for hidden_count in HIDDEN_LAYER_COUNTS:
    architecture = (NEURONS_PER_LAYER,) * hidden_count

    for activation in ACTIVATION_FUNCTIONS:
        for learning_rate in LEARNING_RATES:
            run_number += 1

            print(
                f"\nExperiment {run_number}/{total_experiments}: "
                f"{hidden_count} hidden layers | {activation} | "
                f"learning rate={learning_rate}"
            )

            memory_before = process.memory_info().rss
            wall_start = time.perf_counter()
            cpu_start = time.process_time()

            model = MLPRegressor(
                hidden_layer_sizes=architecture,
                activation=activation,
                solver="adam",
                learning_rate="constant",
                learning_rate_init=learning_rate,
                max_iter=MAX_ITER,
                early_stopping=True,
                validation_fraction=0.10,
                n_iter_no_change=25,
                random_state=RANDOM_STATE
            )

            model.fit(X_train_processed, y_train_scaled)

            pred_scaled = model.predict(X_val_processed)
            y_val_pred = y_scaler.inverse_transform(
                pred_scaled.reshape(-1, 1)
            ).ravel()

            cpu_time = time.process_time() - cpu_start
            wall_time = time.perf_counter() - wall_start
            memory_after = process.memory_info().rss
            memory_change = (memory_after - memory_before) / (1024 ** 2)

            r2 = r2_score(y_val, y_val_pred)
            rmse = np.sqrt(mean_squared_error(y_val, y_val_pred))
            mae = mean_absolute_error(y_val, y_val_pred)

            key = (hidden_count, activation, learning_rate)

            experiment_results.append({
                "Hidden_Layers": hidden_count,
                "Neurons_Per_Layer": NEURONS_PER_LAYER,
                "Architecture": str(architecture),
                "Activation": activation,
                "Learning_Rate": learning_rate,
                "R2": r2,
                "RMSE": rmse,
                "MAE": mae,
                "Wall_Time_s": wall_time,
                "CPU_Time_s": cpu_time,
                "Memory_Change_MB": memory_change,
                "Iterations": model.n_iter_,
                "Converged_Before_Max_Iter": model.n_iter_ < MAX_ITER
            })

            loss_curves[key] = model.loss_curve_
            validation_predictions[key] = y_val_pred

            print(
                f"Validation R2={r2:.4f} | RMSE={rmse:.2f} | MAE={mae:.2f}"
            )
            print(
                f"Wall={wall_time:.2f}s | CPU={cpu_time:.2f}s | "
                f"Memory change={memory_change:.2f} MB"
            )

total_experiment_time = time.perf_counter() - total_start

results = pd.DataFrame(experiment_results)
results.to_csv(
    os.path.join(OUTPUT_DIR, "ann_all_63_experiments.csv"),
    index=False
)


Experiment 1/63: 1 hidden layers | relu | learning rate=0.001
Validation R2=0.8170 | RMSE=5553.37 | MAE=3230.07
Wall=2.11s | CPU=2.45s | Memory change=4.00 MB

Experiment 2/63: 1 hidden layers | relu | learning rate=0.01
Validation R2=0.8164 | RMSE=5562.42 | MAE=3212.58
Wall=0.31s | CPU=0.31s | Memory change=0.02 MB

Experiment 3/63: 1 hidden layers | relu | learning rate=0.1
Validation R2=0.8071 | RMSE=5701.02 | MAE=3819.92
Wall=0.29s | CPU=0.27s | Memory change=0.00 MB

Experiment 4/63: 1 hidden layers | tanh | learning rate=0.001
Validation R2=0.8148 | RMSE=5586.96 | MAE=3127.71
Wall=4.06s | CPU=4.17s | Memory change=0.01 MB

Experiment 5/63: 1 hidden layers | tanh | learning rate=0.01
Validation R2=0.8153 | RMSE=5579.62 | MAE=3267.98
Wall=0.83s | CPU=0.86s | Memory change=0.01 MB

Experiment 6/63: 1 hidden layers | tanh | learning rate=0.1
Validation R2=0.8077 | RMSE=5693.23 | MAE=3346.25
Wall=0.48s | CPU=0.52s | Memory change=0.00 MB

Experiment 7/63: 1 hidden layers | logistic |

In [4]:
best_index = results["RMSE"].idxmin()
best_row = results.loc[best_index]

best_key = (
    int(best_row["Hidden_Layers"]),
    best_row["Activation"],
    float(best_row["Learning_Rate"])
)

print("\nBest validation configuration:")
print(best_row.to_string())


Best validation configuration:
Hidden_Layers                           3
Neurons_Per_Layer                      64
Architecture                 (64, 64, 64)
Activation                       logistic
Learning_Rate                        0.01
R2                               0.830585
RMSE                          5343.335734
MAE                           3064.899462
Wall_Time_s                      2.545141
CPU_Time_s                       3.953125
Memory_Change_MB                 0.003906
Iterations                            272
Converged_Before_Max_Iter            True


In [5]:
best_architecture = (NEURONS_PER_LAYER,) * best_key[0]

final_model = MLPRegressor(
    hidden_layer_sizes=best_architecture,
    activation=best_key[1],
    solver="adam",
    learning_rate="constant",
    learning_rate_init=best_key[2],
    max_iter=MAX_ITER,
    early_stopping=True,
    validation_fraction=0.10,
    n_iter_no_change=25,
    random_state=RANDOM_STATE
)

final_memory_before = process.memory_info().rss
final_wall_start = time.perf_counter()
final_cpu_start = time.process_time()

final_model.fit(X_train_val_processed, y_train_val_scaled)

final_train_wall_time = time.perf_counter() - final_wall_start
final_train_cpu_time = time.process_time() - final_cpu_start
final_memory_after = process.memory_info().rss
final_memory_change = (
    final_memory_after - final_memory_before
) / (1024 ** 2)

test_predict_start = time.perf_counter()
test_pred_scaled = final_model.predict(X_test_processed)
test_prediction_time = time.perf_counter() - test_predict_start

best_prediction = y_scaler.inverse_transform(
    test_pred_scaled.reshape(-1, 1)
).ravel()

test_r2 = r2_score(y_test, best_prediction)
test_rmse = np.sqrt(mean_squared_error(y_test, best_prediction))
test_mae = mean_absolute_error(y_test, best_prediction)

print("\nHeld-out test-set results:")
print(f"R²   : {test_r2:.6f}")
print(f"RMSE : {test_rmse:.6f}")
print(f"MAE  : {test_mae:.6f}")



Held-out test-set results:
R²   : 0.869919
RMSE : 4493.873029
MAE  : 2669.925846


In [6]:
best_summary = pd.DataFrame([{
    "Model": "ANN",
    "Hidden_Layers": best_key[0],
    "Neurons_Per_Layer": NEURONS_PER_LAYER,
    "Activation": best_key[1],
    "Learning_Rate": best_key[2],
    "Validation_R2": best_row["R2"],
    "Validation_RMSE": best_row["RMSE"],
    "Validation_MAE": best_row["MAE"],
    "Test_R2": test_r2,
    "Test_RMSE": test_rmse,
    "Test_MAE": test_mae,
    "Final_Training_Wall_Time_s": final_train_wall_time,
    "Final_Training_CPU_Time_s": final_train_cpu_time,
    "Final_Training_Memory_Change_MB": final_memory_change,
    "Test_Prediction_Time_s": test_prediction_time
}])

best_summary.to_csv(
    os.path.join(OUTPUT_DIR, "ann_best_configuration_metrics.csv"),
    index=False
)

pd.DataFrame({
    "Actual_Expenses": np.asarray(y_test),
    "Predicted_Expenses": best_prediction,
    "Residual": np.asarray(y_test) - best_prediction
}).to_csv(
    os.path.join(OUTPUT_DIR, "ann_best_actual_vs_predicted.csv"),
    index=False
)

In [7]:
METRICS = {
    "RMSE": {
        "column": "RMSE",
        "ylabel": "RMSE (insurance expense units)"
    },
    "MAE": {
        "column": "MAE",
        "ylabel": "MAE (insurance expense units)"
    },
    "R2": {
        "column": "R2",
        "ylabel": "R²"
    }
}


In [8]:
for depth in HIDDEN_LAYER_COUNTS:
    subset = results[results["Hidden_Layers"] == depth].copy()

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    x = np.arange(len(ACTIVATION_FUNCTIONS))
    width = 0.24

    for ax, metric in zip(axes, METRICS):
        for j, lr in enumerate(LEARNING_RATES):
            values = []
            for activation in ACTIVATION_FUNCTIONS:
                row = subset[
                    (subset["Activation"] == activation)
                    & (subset["Learning_Rate"] == lr)
                ]
                values.append(row[METRICS[metric]["column"]].iloc[0])

            ax.bar(
                x + (j - 1) * width,
                values,
                width,
                label=f"LR={lr:g}"
            )

        ax.set_title(metric)
        ax.set_xlabel("Activation Function")
        ax.set_ylabel(METRICS[metric]["ylabel"])
        ax.set_xticks(x)
        ax.set_xticklabels(["ReLU", "Tanh", "Logistic"])
        ax.grid(axis="y", linestyle="--", alpha=0.3)

    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, title="Learning Rate",
               loc="upper center", ncol=3)
    fig.suptitle(
        f"ANN Performance with {depth} Hidden Layer(s)",
        y=1.08, fontsize=14
    )
    fig.tight_layout()
    fig.savefig(
        os.path.join(OUTPUT_DIR, f"{depth:02d}_hidden_layer_{depth}.png"),
        dpi=300, bbox_inches="tight"
    )
    plt.close(fig)


In [9]:
for index, activation in enumerate(ACTIVATION_FUNCTIONS, start=8):
    subset = results[results["Activation"] == activation].copy()

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    for ax, metric in zip(axes, METRICS):
        for lr in LEARNING_RATES:
            part = subset[
                subset["Learning_Rate"] == lr
            ].sort_values("Hidden_Layers")

            ax.plot(
                part["Hidden_Layers"],
                part[METRICS[metric]["column"]],
                marker="o",
                linewidth=1.8,
                label=f"LR={lr:g}"
            )

        ax.set_title(metric)
        ax.set_xlabel("Number of Hidden Layers")
        ax.set_ylabel(METRICS[metric]["ylabel"])
        ax.set_xticks(HIDDEN_LAYER_COUNTS)
        ax.grid(True, linestyle="--", alpha=0.3)

    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, title="Learning Rate",
               loc="upper center", ncol=3)
    fig.suptitle(
        f"ANN Performance - {activation.capitalize()} Activation",
        y=1.08, fontsize=14
    )
    fig.tight_layout()
    fig.savefig(
        os.path.join(OUTPUT_DIR, f"{index:02d}_activation_{activation}.png"),
        dpi=300, bbox_inches="tight"
    )
    plt.close(fig)

In [10]:
best_loss = final_model.loss_curve_

fig, ax = plt.subplots(figsize=(9, 6))
ax.plot(best_loss, linewidth=2)
ax.set_title(
    f"Training Loss - {best_key[0]} Layers, "
    f"{best_key[1].capitalize()}, LR={best_key[2]}"
)
ax.set_xlabel("Training Iteration")
ax.set_ylabel("Training Loss")
ax.grid(True, linestyle="--", alpha=0.35)
fig.tight_layout()
fig.savefig(
    os.path.join(OUTPUT_DIR, "11_best_model_training_loss.png"),
    dpi=300, bbox_inches="tight"
)
plt.close(fig)


In [11]:
actual = np.asarray(y_test)
predicted = np.asarray(best_prediction)

plot_min = min(actual.min(), predicted.min())
plot_max = max(actual.max(), predicted.max())

fig, ax = plt.subplots(figsize=(9, 7))
ax.scatter(
    actual, predicted,
    s=40, alpha=0.6,
    edgecolors="black", linewidths=0.3,
    label="ANN Predictions"
)
ax.plot(
    [plot_min, plot_max], [plot_min, plot_max],
    linestyle="--", linewidth=2,
    label="Perfect Prediction"
)
ax.set_title("Actual vs Predicted Insurance Expenses")
ax.set_xlabel("Actual Expenses")
ax.set_ylabel("Predicted Expenses")
ax.set_xlim(plot_min, plot_max)
ax.set_ylim(plot_min, plot_max)
ax.legend()
ax.grid(True, linestyle="--", alpha=0.3)
fig.tight_layout()
fig.savefig(
    os.path.join(OUTPUT_DIR, "12_actual_vs_predicted.png"),
    dpi=300, bbox_inches="tight"
)
plt.close(fig)

In [12]:
residuals = actual - predicted

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(
    predicted, residuals,
    alpha=0.6, s=30,
    edgecolors="black", linewidths=0.25
)
axes[0].axhline(0, linestyle="--", linewidth=2)
axes[0].set_title("Residuals vs Predicted Expenses")
axes[0].set_xlabel("Predicted Expenses")
axes[0].set_ylabel("Residual (Actual - Predicted)")
axes[0].grid(alpha=0.3)

axes[1].hist(residuals, bins=30, edgecolor="black", alpha=0.75)
axes[1].axvline(0, linestyle="--", linewidth=2)
axes[1].set_title("Distribution of Residual Errors")
axes[1].set_xlabel("Residual (Actual - Predicted)")
axes[1].set_ylabel("Frequency")
axes[1].grid(axis="y", alpha=0.3)

fig.suptitle("ANN Residual Error Analysis")
fig.tight_layout()
fig.savefig(
    os.path.join(OUTPUT_DIR, "13_residual_error_analysis.png"),
    dpi=300, bbox_inches="tight"
)
plt.close(fig)

In [13]:
results.sort_values("RMSE").to_csv(
    os.path.join(OUTPUT_DIR, "ann_ranked_63_experiments.csv"),
    index=False
)

resource_columns = [
    "Hidden_Layers", "Activation", "Learning_Rate",
    "Wall_Time_s", "CPU_Time_s", "Memory_Change_MB",
    "Iterations", "Converged_Before_Max_Iter"
]
results[resource_columns].to_csv(
    os.path.join(OUTPUT_DIR, "ann_resource_comparison.csv"),
    index=False
)




In [16]:
plot_files = (
    [f"{depth:02d}_hidden_layer_{depth}.png"
     for depth in HIDDEN_LAYER_COUNTS]
    + [f"{index:02d}_activation_{activation}.png"
       for index, activation in enumerate(ACTIVATION_FUNCTIONS, start=8)]
    + [
        "11_best_model_training_loss.png",
        "12_actual_vs_predicted.png",
        "13_residual_error_analysis.png"
    ]
)

missing_plots = [
    name for name in plot_files
    if not os.path.exists(os.path.join(OUTPUT_DIR, name))
]

print("\n" + "=" * 75)
print("ANN REGRESSION EXPERIMENT SUMMARY")
print("=" * 75)
print(best_summary.to_string(index=False))
print(f"\nTotal ANN configurations: {len(results)}")
print(f"Total experiment execution time: {total_experiment_time:.2f} seconds")
print(f"Final held-out test prediction time: {test_prediction_time:.4f} seconds")
print(f"\nExpected plot images: {len(plot_files)}")
print("Missing plot files:", missing_plots)
print("\nGenerated plot files:")
for filename in plot_files:
    print("-", filename)

print("\nAll 63 configurations saved in:")
print(os.path.join(OUTPUT_DIR, "ann_all_63_experiments.csv"))
print("Output folder:", OUTPUT_DIR)




ANN REGRESSION EXPERIMENT SUMMARY
Model  Hidden_Layers  Neurons_Per_Layer Activation  Learning_Rate  Validation_R2  Validation_RMSE  Validation_MAE  Test_R2   Test_RMSE    Test_MAE  Final_Training_Wall_Time_s  Final_Training_CPU_Time_s  Final_Training_Memory_Change_MB  Test_Prediction_Time_s
  ANN              3                 64   logistic           0.01       0.830585      5343.335734     3064.899462 0.869919 4493.873029 2669.925846                     0.98893                      1.625                         0.007812                0.002188

Total ANN configurations: 63
Total experiment execution time: 78.57 seconds
Final held-out test prediction time: 0.0022 seconds

Expected plot images: 13
Missing plot files: []

Generated plot files:
- 01_hidden_layer_1.png
- 02_hidden_layer_2.png
- 03_hidden_layer_3.png
- 04_hidden_layer_4.png
- 05_hidden_layer_5.png
- 06_hidden_layer_6.png
- 07_hidden_layer_7.png
- 08_activation_relu.png
- 09_activation_tanh.png
- 10_activation_logistic.png